In [ ]:
import glfw
from OpenGL.GL import *
import warp as wp
import numpy as np

# ... [Insert your provided render code and kernel here] ...
# @event_scope
# def render(m, d, rc): ...

class GlfwWarpViewer:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        
        if not glfw.init():
            raise Exception("GLFW failed")
            
        self.window = glfw.create_window(width, height, "Warp Raytracer", None, None)
        glfw.make_context_current(self.window)
        
        # Create a texture to hold the Warp output
        self.texture = glGenTextures(1)
        glBindTexture(GL_TEXTURE_2D, self.texture)
        glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
        glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    def update_and_show(self, rc):
        """
        Takes the RenderContext (rc) which has been filled by your 
        custom render() function and puts it on screen.
        """
        
        # 1. Get the data from your RenderContext
        # Your kernel writes to rc.rgb_data (assumed flat array of uint32)
        # We need to reshape it to (H, W, 4) for OpenGL
        
        # shape: (num_pixels,) -> (H, W, 4)
        # The uint32 contains packed RGBA. We view it as byte (uint8)
        pixels_uint32 = rc.rgb_data.numpy() # Copy GPU -> CPU
        pixels_uint8 = pixels_uint32.view(dtype=np.uint8).reshape(self.height, self.width, 4)

        # 2. Upload to OpenGL Texture
        glBindTexture(GL_TEXTURE_2D, self.texture)
        # GL_RGBA because your kernel packs 4 channels
        glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, self.width, self.height, 0, GL_RGBA, GL_UNSIGNED_BYTE, pixels_uint8)

        # 3. Draw Texture to Screen
        glfw.make_context_current(self.window)
        glClear(GL_COLOR_BUFFER_BIT)
        glEnable(GL_TEXTURE_2D)
        
        glBegin(GL_QUADS)
        glTexCoord2f(0.0, 1.0); glVertex2f(-1.0, -1.0)
        glTexCoord2f(1.0, 1.0); glVertex2f( 1.0, -1.0)
        glTexCoord2f(1.0, 0.0); glVertex2f( 1.0,  1.0)
        glTexCoord2f(0.0, 0.0); glVertex2f(-1.0,  1.0)
        glEnd()
        
        glfw.swap_buffers(self.window)
        glfw.poll_events()
        
    def is_running(self):
        return not glfw.window_should_close(self.window)
    
    def close(self):
        glfw.terminate()

# --- MAIN LOOP USAGE ---

# 1. Setup
viewer = GlfwWarpViewer(CAM_RES[0], CAM_RES[1])

while viewer.is_running():
    # 2. Physics Step
    # mjw.step(model, data) ...
    
    # 3. Update BVH (Crucial for raytracing!)
    mjw.refit_bvh(model, data, rc)
    
    # 4. RUN YOUR KERNEL
    # This executes the huge code block you provided on the GPU
    render(model, data, rc) 
    
    # 5. Display the result
    viewer.update_and_show(rc)

viewer.close()

In [ ]:
from OpenGL.GL import *
import numpy as np
import glfw

# ... (Assuming your window creation and texture generation are above) ...

pixels_uint32 = rc.rgb_data.numpy() # Copy GPU -> CPU
# First, reshape to your standard RGB (3 channels)
pixels_rgb = pixels_uint32.view(dtype=np.uint8).reshape(gl_height, gl_width, 3)

# --- 1. MANUALLY ADD ALPHA CHANNEL ---
# Convert 0.7 alpha to a uint8 value (0.7 * 255 ≈ 178)
alpha_value = int(0.7 * 255) 

# Create an alpha channel array of shape (height, width, 1) filled with 178
alpha_channel = np.full((gl_height, gl_width, 1), alpha_value, dtype=np.uint8)

# Concatenate along the color axis (axis=2) to make shape (height, width, 4)
pixels_rgba = np.concatenate((pixels_rgb, alpha_channel), axis=2)


# --- 2. UPLOAD TO OPENGL TEXTURE ---
glBindTexture(GL_TEXTURE_2D, texture)

# Ensure default 4-byte alignment (RGBA is 4 bytes, so this is safe and standard)
glPixelStorei(GL_UNPACK_ALIGNMENT, 4)

# Upload the new 4-channel data using GL_RGBA
glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, gl_width, gl_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, pixels_rgba)


# --- 3. DRAW TEXTURE TO SCREEN ---
glfw.make_context_current(window)

# Optional: Set a background clear color (e.g., dark gray) so you can actually 
# see the transparency working against the background.
glClearColor(0.2, 0.2, 0.2, 1.0) 
glClear(GL_COLOR_BUFFER_BIT)

# CRITICAL: Enable blending so OpenGL actually uses the Alpha channel!
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)

glEnable(GL_TEXTURE_2D)

glBegin(GL_QUADS)
glTexCoord2f(0.0, 1.0); glVertex2f(-1.0, -1.0)
glTexCoord2f(1.0, 1.0); glVertex2f( 1.0, -1.0)
glTexCoord2f(1.0, 0.0); glVertex2f( 1.0,  1.0)
glTexCoord2f(0.0, 0.0); glVertex2f(-1.0,  1.0)
glEnd()

glfw.swap_buffers(window)
glfw.poll_events()

In [ ]:
import glfw
import mujoco
import warp as wp
import numpy as np

# [Insert your provided 'render' function and kernel here]
# ...

class GlfwWarpViewer:
    def __init__(self, width, height, model):
        self.width = width
        self.height = height
        
        if not glfw.init():
            raise Exception("GLFW failed")
            
        # 1. Create Window
        self.window = glfw.create_window(width, height, "MuJoCo Warp (No OpenGL Import)", None, None)
        glfw.make_context_current(self.window)
        
        # 2. Initialize MuJoCo Render Context
        # MuJoCo will handle the OpenGL setup internally here
        self.mjr_context = mujoco.MjrContext(model, mujoco.mjtFontScale.mjFONTSCALE_150)
        
        # 3. Create a viewport (the area of the window to draw into)
        self.viewport = mujoco.MjrRect(0, 0, width, height)

    def update_and_show(self, rc):
        # --- A. PREPARE IMAGE ---
        # 1. Copy Warp GPU data -> CPU Numpy
        # format: (H * W,) flat uint32 array of packed RGBA
        pixels_uint32 = rc.rgb_data.numpy()
        
        # 2. Convert to RGBA (uint8)
        # View as uint8 (4 bytes per pixel) -> Reshape to (H, W, 4)
        pixels_rgba = pixels_uint32.view(dtype=np.uint8).reshape(self.height, self.width, 4)
        
        # 3. Extract just RGB (MuJoCo prefers RGB for drawing usually)
        pixels_rgb = pixels_rgba[:, :, :3]
        
        # 4. Flip Vertically
        # OpenGL (and MuJoCo) origin is Bottom-Left. 
        # Image memory origin is Top-Left. We must flip.
        pixels_rgb_flipped = np.flipud(pixels_rgb)

        # --- B. DRAW USING MUJOCO ---
        glfw.make_context_current(self.window)
        
        # This function is the magic. It replaces all the OpenGL code.
        # It takes the numpy array and draws it to the screen.
        mujoco.mjr_drawPixels(pixels_rgb_flipped, None, self.viewport, self.mjr_context)

        # --- C. SWAP BUFFERS ---
        glfw.swap_buffers(self.window)
        glfw.poll_events()
        
    def is_running(self):
        return not glfw.window_should_close(self.window)
    
    def close(self):
        self.mjr_context.free() # Good practice to free MuJoCo context
        glfw.terminate()

# --- USAGE ---

# Assuming 'model', 'data', 'rc' are already set up
viewer = GlfwWarpViewer(CAM_RES[0], CAM_RES[1], model)

while viewer.is_running():
    # 1. Physics & Raytracing (Warp)
    mjw.refit_bvh(model, data, rc)
    render(model, data, rc) # Your custom render function
    
    # 2. Display (MuJoCo/GLFW)
    viewer.update_and_show(rc)

viewer.close()